# Retreiver Demonstration

In [50]:
# Importing packages
import pandas as pd
import torch
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, logging, AutoModel
logging.set_verbosity_error()
import numpy as np
from concurrent.futures import ThreadPoolExecutor
from dotenv import load_dotenv
import os
from torch import Tensor
import faiss 
import json
from beir.datasets.data_loader import GenericDataLoader
import torch.nn.functional as F

In [2]:
# Loading token
load_dotenv('token.env')
token = os.getenv('HUGGINGFACE_TOKEN')

# Loading model - pass token directly
model_name = "meta-llama/Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, token=token)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="auto",
    token=token  # Pass token here
)

In [27]:
# Loading Queries:
data_dir = "/work/mbouthil/projects/research_project/RAG/datasets/msmarco"
corpus, queries, qrels = GenericDataLoader(data_folder=data_dir).load(split="train")

  0%|          | 0/8841823 [00:00<?, ?it/s]

### Testing Passage:

In [69]:
for keys, values in corpus.items():
    print(keys)
    print(values)
    break

0
{'text': 'The presence of communication amid scientific minds was equally important to the success of the Manhattan Project as scientific intellect was. The only cloud hanging over the impressive achievement of the atomic researchers and engineers is what their success truly meant; hundreds of thousands of innocent lives obliterated.', 'title': ''}


In [33]:
corpus['250']

{'text': 'With its microwave receptors at 338 m (1,109 ft.) and at the 553.33m (1,815 ft., 5 inches) antenna, the CN Tower swiftly solved the communications problems with room to spare and as a result, people living in the Toronto area now enjoy some of the clearest reception in North America.',
 'title': ''}

### Example of a Query:

In [4]:
i = 5

ex_q = [(q_id, queries[q_id]) for q_id in queries][i]
print(ex_q)

('312651', 'how much does an average person make for tutoring')


### Linking Query to Passage

In [5]:
passage_id = list(qrels[ex_q[0]].keys())
print(passage_id)

['616']


### Gathering the Passage

In [6]:
print(corpus[passage_id[0]]['text'])

In-home tutors can earn anywhere from $10 to $80 an hour, depending on the type of lesson, the studentâs skill and age level and the tutorâs experience. Tutors often charge more for older students or those who require more advanced lessons.


# Loading the vector DB, index and json

In [7]:
# Loading Index
index = faiss.read_index("/work/mbouthil/projects/research_project/RAG/retrieval_data/passage.index")

# Load Metadata
metadata = []
with open("/work/mbouthil/projects/research_project/RAG/retrieval_data/passage_metadata.jsonl") as f:
    for line in f:
        metadata.append(json.loads(line))

### Loading the Trained Query Encoder

In [51]:
# Loading Query Encoder
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
query_encoder = AutoModel.from_pretrained(
    "/work/mbouthil/projects/research_project/RAG/model_weights/query_encoder"
) # .to("cuda")

query_encoder.eval()

def encode_query(query:str, batch_size:int=32) -> Tensor:

    embeddings = []

    with torch.no_grad():
        inputs = tokenizer(
            query, 
            padding=True,
            truncation=True,
            return_tensors="pt",
            max_length=32
        ) #.to("cuda")

    emb = query_encoder(**inputs).last_hidden_state[:, 0]
    emb = F.normalize(emb, p=2, dim=-1)

    embeddings.append(emb.cpu())

    return torch.cat(embeddings, 0)

### Embedding the Quesetion

In [52]:
question = ex_q[1]
print(question)

# Embedding Query
q_emb = encode_query([question]).detach().cpu().numpy()
print('Embedding shape: ', q_emb.shape)

how much does an average person make for tutoring
Embedding shape:  (1, 768)


In [58]:
# Matching for the top K=10 highest scores
K = 50
scores, ids = index.search(q_emb, K)
# candidates = [metadata[i]["text"] for i in ids[0]]
# test_df = pd.DataFrame({'scores': scores, 'passage':candidates})
# test_df.head(10)

In [65]:
type(metadata)

list

In [61]:
for id in ids[0]:
    p_id = metadata[str(id)]
    print(corpus[str(p_id)]['text'])
    # print(id)

TypeError: list indices must be integers or slices, not str

In [41]:
for id in ids[0]:
    print(str(id))
    print("\n")

185928


860681


7069679


5161650


5591496


1158463


455701


156404


476120


1401997




In [15]:
K = 10
scores, ids = index.search(q_emb, K)

In [35]:
corpus[str(1000)]

{'text': 'QuickFacts Matanuska-Susitna Borough, Alaska; UNITED STATES QuickFacts provides statistics for all states and counties, and for cities and towns with a population of 5,000 or more.',
 'title': ''}

# Generation